# 07 – SHAP Objašnjenja

Koristimo SHAP (SHapley Additive exPlanations) da objasnimo zašto LightGBM model daje određene predikcije.

Analize:
1. Globalna važnost feature-a (summary + bar)
2. Važnost po grupama feature-a
3. Dependence plotovi za top feature-e
4. Waterfall plot – objašnjenje pojedinačnih predikcija
5. SHAP po kategoriji proizvoda (family)
6. Snimanje SHAP vrijednosti za web app

In [ ]:
import sys, subprocess
try:
    import shap
    print(f'shap {shap.__version__} već instaliran.')
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'shap', '-q'], check=True)
    import shap
    print(f'shap {shap.__version__} instaliran.')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import pickle
import shap
import warnings
warnings.filterwarnings('ignore')

PROCESSED = '../data/processed/'
MODELS    = '../models/'

# Grupe feature-a za preglednost
FEATURE_GROUPS = {
    'Historijski lagovi':   ['lag_7', 'lag_14', 'lag_28', 'lag_56'],
    'Rolling statistike':   ['roll_mean_7', 'roll_mean_14', 'roll_mean_28',
                             'roll_std_7',  'roll_std_14',  'roll_std_28'],
    'Prophet':              ['prophet_trend', 'prophet_weekly', 'prophet_yearly', 'prophet_yhat'],
    'Kalendar':             ['year', 'month', 'weekofyear', 'dayofweek', 'dayofmonth',
                             'is_weekend', 'is_payday', 'is_national_holiday', 'is_local_holiday'],
    'Prodavnica/kategorija':['store_nbr', 'family_enc', 'type_enc', 'city_enc', 'state_enc', 'cluster'],
    'Eksterni faktori':     ['onpromotion', 'transactions', 'oil_price', 'days_after_earthquake'],
}

print('Spreman.')

## 1. Učitaj model i podatke

In [ ]:
with open(MODELS + 'lgbm_prophet.pkl', 'rb') as f:
    lgbm_data = pickle.load(f)

model    = lgbm_data['model']
features = lgbm_data['features']

val_df = pd.read_parquet(PROCESSED + 'val_features.parquet')
X_val  = val_df[features]

print(f'Val: {X_val.shape}')
print(f'Features: {len(features)}')

## 2. Izračunaj SHAP vrijednosti

In [ ]:
explainer   = shap.TreeExplainer(model)
shap_values = explainer(X_val)

print(f'SHAP values shape: {shap_values.values.shape}')
print(f'Bazna vrijednost (expected_value): {explainer.expected_value:.4f}')
print(f'  → u sales prostoru: {np.expm1(explainer.expected_value):.2f}')

## 3. Globalna važnost – Summary plot

In [ ]:
# Uzorak za plotove (puni val je 26730 redova – može biti sporo za beeswarm)
np.random.seed(42)
sample_idx = np.random.choice(len(X_val), size=3000, replace=False)
X_sample   = X_val.iloc[sample_idx]
sv_sample  = shap.Explanation(
    values          = shap_values.values[sample_idx],
    base_values     = shap_values.base_values[sample_idx],
    data            = shap_values.data[sample_idx],
    feature_names   = features,
)

plt.figure(figsize=(10, 8))
shap.summary_plot(sv_sample, X_sample, show=False, max_display=20)
plt.title('SHAP Summary – top 20 feature-a (uzorak 3000 redova)', fontsize=13)
plt.tight_layout()
plt.savefig(MODELS + 'shap_summary.png', dpi=120, bbox_inches='tight')
plt.show()
print('Sačuvano: shap_summary.png')

## 4. Bar plot – mean |SHAP|

In [ ]:
mean_shap = pd.Series(
    np.abs(shap_values.values).mean(axis=0),
    index=features
).sort_values(ascending=False)

# Boja po grupi
feat_to_group = {f: g for g, fs in FEATURE_GROUPS.items() for f in fs}
group_colors = {
    'Historijski lagovi':    '#2196F3',
    'Rolling statistike':    '#03A9F4',
    'Prophet':               '#9C27B0',
    'Kalendar':              '#FF9800',
    'Prodavnica/kategorija': '#4CAF50',
    'Eksterni faktori':      '#F44336',
}
colors = [group_colors.get(feat_to_group.get(f, ''), '#999') for f in mean_shap.index]

fig, ax = plt.subplots(figsize=(9, 10))
bars = ax.barh(mean_shap.index[::-1], mean_shap.values[::-1], color=colors[::-1])
ax.set_xlabel('mean |SHAP vrijednost| (log-prodaja prostor)')
ax.set_title('Globalna važnost feature-a')

# Legenda grupa
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=g) for g, c in group_colors.items()]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig(MODELS + 'shap_bar.png', dpi=120, bbox_inches='tight')
plt.show()

print('Top 10 feature-a:')
for feat, val in mean_shap.head(10).items():
    group = feat_to_group.get(feat, '?')
    print(f'  {feat:<25} {val:.4f}  [{group}]')

## 5. Važnost po grupama feature-a

In [ ]:
group_importance = {}
for group, feats in FEATURE_GROUPS.items():
    idxs = [features.index(f) for f in feats if f in features]
    group_importance[group] = np.abs(shap_values.values[:, idxs]).mean()

gi = pd.Series(group_importance).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(gi.index, gi.values,
               color=[group_colors[g] for g in gi.index])
ax.set_xlabel('mean |SHAP| po grupi')
ax.set_title('Doprinos svake grupe feature-a')

for bar, val in zip(bars, gi.values):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(MODELS + 'shap_groups.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Dependence plotovi – top 4 feature-a

In [ ]:
top4 = mean_shap.head(4).index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()

for i, feat in enumerate(top4):
    feat_idx = features.index(feat)
    x_vals   = X_val[feat].values[sample_idx]
    s_vals   = shap_values.values[sample_idx, feat_idx]

    sc = axes[i].scatter(x_vals, s_vals, alpha=0.3, s=5, c=s_vals, cmap='coolwarm')
    axes[i].axhline(0, color='black', linewidth=0.8, linestyle='--')
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('SHAP vrijednost')
    axes[i].set_title(f'Dependence: {feat}')
    plt.colorbar(sc, ax=axes[i], shrink=0.8)

plt.suptitle('SHAP Dependence plotovi – top 4 feature-a', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(MODELS + 'shap_dependence.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Waterfall plot – objašnjenje pojedinačnih predikcija

In [ ]:
# Predikcije vs stvarno
y_val_log  = val_df['sales_log'].values
y_pred_log = model.predict(X_val, num_iteration=model.best_iteration)
errors     = np.abs(y_pred_log - y_val_log)

# Primjer 1: tipična predikcija (medijalna greška)
median_idx  = np.argsort(errors)[len(errors) // 2]
# Primjer 2: najveća greška
worst_idx   = np.argmax(errors)

for label, idx in [('Tipična predikcija', median_idx), ('Najveća greška', worst_idx)]:
    row = val_df.iloc[idx]
    print(f'\n--- {label} ---')
    print(f'  Prodavnica: {row["store_nbr"]}, Kategorija: {row["family"]}, Datum: {row["date"].date()}')
    print(f'  Stvarna prodaja:    {np.expm1(y_val_log[idx]):.1f}')
    print(f'  Predvidjena prodaja:{np.expm1(y_pred_log[idx]):.1f}')

    fig, ax = plt.subplots(figsize=(10, 5))
    shap.waterfall_plot(
        shap.Explanation(
            values        = shap_values.values[idx],
            base_values   = shap_values.base_values[idx],
            data          = shap_values.data[idx],
            feature_names = features,
        ),
        max_display=12,
        show=False
    )
    plt.title(f'{label} | Store {int(row["store_nbr"])}, {row["family"]}, {row["date"].date()}\n'
              f'Stvarno: {np.expm1(y_val_log[idx]):.1f}  |  Predvidjeno: {np.expm1(y_pred_log[idx]):.1f}',
              fontsize=10)
    plt.tight_layout()
    fname = MODELS + f'shap_waterfall_{label.split()[0].lower()}.png'
    plt.savefig(fname, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'  Sačuvano: {fname}')

## 8. SHAP po kategoriji proizvoda (family)

In [ ]:
# Prosječni |SHAP| top 8 feature-a, po family
top8 = mean_shap.head(8).index.tolist()
top8_idx = [features.index(f) for f in top8]

shap_df = pd.DataFrame(
    np.abs(shap_values.values[:, top8_idx]),
    columns=top8
)
shap_df['family'] = val_df['family'].values

family_shap = shap_df.groupby('family')[top8].mean()

fig, ax = plt.subplots(figsize=(14, 9))
family_shap.plot(kind='bar', ax=ax, colormap='tab10', width=0.8)
ax.set_xlabel('Kategorija (family)')
ax.set_ylabel('mean |SHAP|')
ax.set_title('Važnost top 8 feature-a po kategoriji proizvoda')
ax.legend(loc='upper right', fontsize=8, ncol=2)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig(MODELS + 'shap_by_family.png', dpi=120, bbox_inches='tight')
plt.show()

## 9. Sačuvaj SHAP vrijednosti za web app

In [ ]:
# Sačuvaj mean |SHAP| po feature-u (globalna važnost)
mean_shap.reset_index().rename(columns={'index': 'feature', 0: 'mean_abs_shap'}).to_parquet(
    MODELS + 'shap_importance.parquet', index=False
)

# Sačuvaj mean |SHAP| po feature-u i family-u
family_shap.reset_index().to_parquet(MODELS + 'shap_by_family.parquet', index=False)

# Sačuvaj SHAP vrijednosti za val set (sample 5000 za web app)
sample5k = np.random.choice(len(X_val), size=5000, replace=False)
shap_sample_df = pd.DataFrame(
    shap_values.values[sample5k],
    columns=features
)
shap_sample_df['family']    = val_df['family'].iloc[sample5k].values
shap_sample_df['store_nbr'] = val_df['store_nbr'].iloc[sample5k].values
shap_sample_df['date']      = val_df['date'].iloc[sample5k].values
shap_sample_df.to_parquet(MODELS + 'shap_sample.parquet', index=False)

print('Sačuvano:')
print('  shap_importance.parquet  – globalna važnost feature-a')
print('  shap_by_family.parquet   – važnost po kategoriji')
print('  shap_sample.parquet      – SHAP vrijednosti (sample 5000)')
print()
print('Slike:')
print('  shap_summary.png, shap_bar.png, shap_groups.png')
print('  shap_dependence.png, shap_by_family.png')
print('  shap_waterfall_tipična.png, shap_waterfall_najveća.png')